In [ ]:
# -*- coding: utf-8 -*-
"""fooling.ipynb

In [ ]:
Automatically generated by Colaboratory.

In [ ]:
Original file is located at
    https://colab.research.google.com/drive/1QY20ii6VicpnF4affHo0qU91AbdGVQ0B

 Fooling neural networks

In [ ]:
Here we show how to fool a neural network using a gradient ascent technique over the input.
"""

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input, decode_predictions
from tensorflow.keras import backend as K
from tensorflow.keras import losses 
import numpy as np
import matplotlib.pyplot as plt


Let us start importing the VGG16 model.
<br>
model = VGG16(weights='imagenet', include_top=True)<br>
#model.summary()<br>
ow, we load an image (in our case, an elephant)


In [ ]:
from google.colab import files
uploaded = files.upload()


Next, we classify it. <br>
VGG16 is higly confident it is an elephant.<br>


In [ ]:
img = image.load_img('elephant2.jpg', target_size=(224, 224))

In [ ]:
x0 = image.img_to_array(img)
x = np.expand_dims(x0, axis=0)
preds = model.predict(x)
print("label = {}".format(np.argmax(preds)))
print('Predicted:', decode_predictions(preds, top=3)[0])

In [ ]:
xd = image.array_to_img(x[0])
imageplot = plt.imshow(xd)
plt.show()


Now we try to convert the image into something different: a tiger shark, with label 3.
<br>
output_index = 3 #tiger shark<br>
expected_output = np.zeros(1000)<br>
expected_output[output_index] = 1<br>
expected_output = K.variable(np.reshape(expected_output,(1,1000)))<br>
ow we simply iterate the gradient ascent technique for a sufficent number of steps, working on a copy of the original image


In [ ]:
input_img_data = np.copy(x)

run gradient ascent for 50 steps

In [ ]:
for i in range(70):
    print("iteration n. {}".format(i))
    with tf.GradientTape() as g:
      x = K.variable(input_img_data)
      y = model(x)
      loss = tf.keras.losses.categorical_crossentropy(y,expected_output)
    res = y[0]
    print("elephant prediction: {}".format(res[386]))
    print("tiger shark prediction: {}".format(res[3]))
    grads_value = g.gradient(loss, x)[0]
    print(grads_value.shape)
    ming = np.min(grads_value)
    maxg = np.max(grads_value)
    #print("min grad = {}".format(ming))
    #print("max grad = {}".format(maxg))
    scale = 1/(maxg -ming)
    #brings gradients to a sensible value
    input_img_data = input_img_data - grads_value * scale


At the end, VGG16 is extremely confident he is looking at a tiger shark
<br>
preds = model.predict(input_img_data)<br>
print("label = {}".format(np.argmax(preds)))<br>
print('Predicted:', decode_predictions(preds, top=3)[0])<br>
et us look at the resulting image (we both print the original and the processed image)


In [ ]:
nimg = input_img_data[0]
nimg = image.array_to_img(img)

In [ ]:
plt.figure(figsize=(10,5))
ax = plt.subplot(1, 2, 1)
plt.title("elephant")
ax.get_xaxis().set_visible(False)
ax.get_yaxis().set_visible(False)
plt.imshow(xd)
ax = plt.subplot(1, 2, 2)
plt.imshow(nimg)
plt.title("tiger shark")
ax.get_xaxis().set_visible(False)
ax.get_yaxis().set_visible(False)

In [ ]:
imageplot = plt.imshow(img)
plt.show()


We just fooled the neural network! 
